# 04 — Frontier-Vergleich (Phase 5)

Hand-Annotation aus Phase 2 (`annotation/meine_gold.csv`) gegen Frontier-LLM-Annotation derselben 12 Anzeigen — **zwei Modelle im Vergleich**: ChatGPT (`frontier_predictions_chatgpt.json`) und Claude (`frontier_predictions_claude.json`).

Output: pro Modell eine CSV (`annotation/frontier_gold_chatgpt.csv` / `..._claude.csv`), κ-Tabelle Mensch ↔ Frontier, Disagreements gegenüber dem Hand-Gold, direkte Gegenüberstellung der drei Quellen sowie drei konkrete Disagreement-Beispiele. Material fürs Make-or-Buy-Memo (`memo_make_or_buy.md` im Repo-Root).

Cheatsheet: `CHEATSHEETS/frontier-llm-workflow.md`.


## Run-Header

| Feld | Wert |
|---|---|
| Datum | 2026-06-04 |
| Frontier-Modell 1 | ChatGPT (Version selbst eintragen) |
| Frontier-Modell 2 | Claude (Version selbst eintragen) |
| Prompt-Variante | identischer Prompt für beide Modelle, Schema + 3 Few-Shots (Eingabe-Block: `notebooks/frontier_input_block.txt`) |
| Anzahl Korrektur-Turns | _ (selbst eintragen) |
| Schema-Verletzungs-Mapping | 0 Mappings bei ChatGPT, 0 bei Claude (Frontier-Output war schemakonform) |
| ChatGPT | `frontier_predictions_chatgpt.json` → `annotation/frontier_gold_chatgpt.csv` |
| Claude | `frontier_predictions_claude.json` → `annotation/frontier_gold_claude.csv` |
| Eigene Gold-CSV | `annotation/meine_gold.csv` |


## Daten laden + Schema-Konformität prüfen

Die Zellen suchen robust in typischen Notebook-/Repo-Ordnern. Für deine Handannotation wird bevorzugt `annotations/mein_gold.csv` verwendet; ältere Pfade wie `annotation/meine_gold.csv` funktionieren weiterhin als Fallback.


In [ ]:
import csv
import json
import re
from collections import Counter
from pathlib import Path
from textwrap import dedent

import pandas as pd

# ── Robuste Pfade ─────────────────────────────────────────────────────────
# Das Notebook kann aus /notebooks, dem Repo-Root oder einer Notebook-UI laufen.
CWD = Path.cwd().resolve()
SEARCH_DIRS = [
    CWD,
    CWD.parent,
    CWD / "annotations",
    CWD.parent / "annotations",
    CWD / "annotation",
    CWD.parent / "annotation",
    CWD / "daten",
    CWD.parent / "daten",
    Path("/mnt/data"),  # nur relevant in ChatGPT/Sandbox, lokal harmlos
]

def find_existing_file(*names, required=False):
    """Suche eine Datei in typischen Repo-/Notebook-Ordnern."""
    candidates = []
    for name in names:
        p = Path(name)
        candidates.append(p)
        if p.is_absolute() and p.exists():
            return p.resolve()
        for base in SEARCH_DIRS:
            candidate = base / name
            candidates.append(candidate)
            if candidate.exists():
                return candidate.resolve()
    if required:
        msg = "Datei nicht gefunden. Geprüfte Kandidaten:\n" + "\n".join(f"  - {p}" for p in candidates)
        raise FileNotFoundError(msg)
    return Path(names[0])

# Bevorzugter Pfad laut Aufgabenstellung: annotations/mein_gold.csv
# Fallbacks bleiben drin, damit alte Workshop-Dateinamen weiter funktionieren.
GOLD_PATH = find_existing_file(
    "annotations/mein_gold.csv",
    "../annotations/mein_gold.csv",
    "mein_gold.csv",
    "annotation/mein_gold.csv",
    "../annotation/mein_gold.csv",
    "annotation/meine_gold.csv",
    "../annotation/meine_gold.csv",
    "meine_gold.csv",
)
KORPUS_PATH = find_existing_file(
    "eigener_korpus.jsonl",
    "daten/eigener_korpus.jsonl",
    "../daten/eigener_korpus.jsonl",
)

PREDICTION_FILES = {
    "chatgpt": find_existing_file(
        "frontier_predictions_chatgpt.json",
        "annotations/frontier_predictions_chatgpt.json",
        "annotation/frontier_predictions_chatgpt.json",
        "../frontier_predictions_chatgpt.json",
    ),
    "claude": find_existing_file(
        "frontier_predictions_claude.json",
        "annotations/frontier_predictions_claude.json",
        "annotation/frontier_predictions_claude.json",
        "../frontier_predictions_claude.json",
    ),
}

# CSV-Ausgaben: bevorzugt in denselben Ordner wie mein_gold.csv,
# sonst in ./annotations anlegen.
if GOLD_PATH.exists():
    ANNOTATION_DIR = GOLD_PATH.parent
else:
    for candidate_dir in [CWD / "annotations", CWD.parent / "annotations", CWD / "annotation", CWD.parent / "annotation"]:
        if candidate_dir.exists():
            ANNOTATION_DIR = candidate_dir
            break
    else:
        ANNOTATION_DIR = CWD / "annotations"
ANNOTATION_DIR.mkdir(parents=True, exist_ok=True)

FRONTIER_CSV_PATHS = {
    "chatgpt": ANNOTATION_DIR / "frontier_gold_chatgpt.csv",
    "claude": ANNOTATION_DIR / "frontier_gold_claude.csv",
}

print("Gefundene Dateien:")
print(f"  Gold:   {GOLD_PATH if GOLD_PATH.exists() else 'NICHT GEFUNDEN'}")
print(f"  Korpus: {KORPUS_PATH if KORPUS_PATH.exists() else 'NICHT GEFUNDEN'}")
for model, path in PREDICTION_FILES.items():
    print(f"  {model}: {path if path.exists() else 'NICHT GEFUNDEN'}")

# Gold/Korpus nur laden, wenn vorhanden. Für reine JSON→CSV-Konvertierung sind sie nicht nötig.
gold_df = pd.read_csv(GOLD_PATH) if GOLD_PATH.exists() else None
korpus = pd.read_json(KORPUS_PATH, lines=True) if KORPUS_PATH.exists() else None

if gold_df is not None:
    # Workshop-Dateien heißen manchmal id, manchmal refnr. Intern nutzen wir beides robust.
    if "id" not in gold_df.columns and "refnr" in gold_df.columns:
        gold_df = gold_df.rename(columns={"refnr": "id"})
    if "refnr" not in gold_df.columns and "id" in gold_df.columns:
        gold_df["refnr"] = gold_df["id"]
    if "id" not in gold_df.columns:
        raise KeyError("Die Hand-Gold-Datei braucht eine Spalte 'id' oder 'refnr'.")

    gold_df["id"] = gold_df["id"].astype(str)
    gold_df["refnr"] = gold_df["refnr"].astype(str)
    gold_ids = gold_df["id"].tolist()
    print(f"\nGold-Records geladen: {len(gold_df)}")
else:
    gold_ids = []
    print("\nHinweis: Ohne annotations/mein_gold.csv bzw. annotation/meine_gold.csv wird nur JSON→CSV erzeugt; κ-Vergleich wird übersprungen.")

if korpus is not None and gold_ids:
    anzeigen = korpus.set_index("refnr").loc[gold_ids]
else:
    anzeigen = None



In [ ]:
# Beide Prediction-JSONs → annotation/frontier_gold_chatgpt.csv und annotation/frontier_gold_claude.csv

SCHEMA_VERLETZUNG_MAP = {
    "homeoffice": {
        "möglich": "teilweise", "moeglich": "teilweise",
        "nach absprache": "teilweise", "flexibel": "teilweise",
        "mobiles arbeiten": "teilweise", "hybrid": "teilweise",
        "kein homeoffice": "nein", "vollzeit remote": "remote",
        "100% remote": "remote", "100 % remote": "remote",
        "vollständig remote": "remote", "ortsunabhängig": "remote",
    },
    "vertragsart": {
        "freelance": "sonstiges", "freiberuflich": "sonstiges",
        "selbstaendig": "sonstiges", "selbständig": "sonstiges",
        "lehrbeauftragter": "sonstiges", "lehrbeauftragte": "sonstiges",
        "leiharbeit": "sonstiges", "trainee": "sonstiges",
    },
    "erfahrungslevel": {
        "berufseinsteiger": "junior", "entry": "junior", "anfänger": "junior",
        "mittel": "mid", "intermediate": "mid",
        "expert": "senior", "experte": "senior",
        "alle level": "egal", "egal welches level": "egal",
    },
    "gehalt_zeitraum": {
        "month": "monat", "monatlich": "monat", "monatsgehalt": "monat",
        "year": "jahr", "jährlich": "jahr", "jaehrlich": "jahr", "jahresgehalt": "jahr",
    },
}

VALID_VALUES = {
    "homeoffice": {"ja", "teilweise", "nein", "remote", "nicht_genannt"},
    "vertragsart": {"ausbildung", "festanstellung", "praktikum", "werkstudent", "sonstiges"},
    "erfahrungslevel": {"junior", "mid", "senior", "egal", "nicht_genannt"},
    "gehalt_zeitraum": {"monat", "jahr"},
}

FELDER_CSV = [
    "id", "homeoffice", "vertragsart", "erfahrungslevel",
    "gehalt_min_eur", "gehalt_zeitraum", "skills_top3", "notiz"
]

def extract_json_array(path: Path):
    raw = path.read_text(encoding="utf-8").strip()
    try:
        data = json.loads(raw)
    except json.JSONDecodeError:
        # Falls ein Modell doch Begleittext geschrieben hat: erstes JSON-Array herauspulen
        match = re.search(r"\[.*\]", raw, re.DOTALL)
        if not match:
            raise ValueError(f"Kein JSON-Array in {path} gefunden — Output prüfen.")
        data = json.loads(match.group(0))
    if not isinstance(data, list):
        raise TypeError(f"{path} enthält kein JSON-Array.")
    return data

def map_value(feld, val):
    if val is None:
        return None
    if isinstance(val, (list, int, float)):
        return val
    s = str(val).strip()
    s_lower = s.lower()
    return SCHEMA_VERLETZUNG_MAP.get(feld, {}).get(s_lower, s_lower)

def normalize_salary(v):
    if v is None or v == "":
        return ""
    try:
        return int(float(str(v).replace(".", "").replace(",", ".")))
    except ValueError:
        return str(v).strip()

def normalize_skills(v):
    if v is None:
        return ""
    if isinstance(v, list):
        return "|".join(str(x).strip() for x in v[:3] if str(x).strip())
    return str(v).strip()

def records_to_csv_rows(records):
    mapped_count = 0
    csv_rows = []
    problems = []

    for idx, rec in enumerate(records, start=1):
        if not isinstance(rec, dict):
            problems.append(f"Record {idx}: kein Objekt")
            continue

        rid = rec.get("id")
        row = {"id": rid, "notiz": ""}

        for feld in ["homeoffice", "vertragsart", "erfahrungslevel", "gehalt_zeitraum"]:
            original = rec.get(feld)
            mapped = map_value(feld, original)

            if isinstance(original, str) and original.strip().lower() != str(mapped).lower():
                mapped_count += 1

            # Wenn Gehalt fehlt, muss Zeitraum leer sein.
            if feld == "gehalt_zeitraum" and rec.get("gehalt_min_eur") in (None, ""):
                mapped = None

            if mapped is not None and feld in VALID_VALUES and mapped not in VALID_VALUES[feld]:
                problems.append(f"{rid}: ungültiger Wert {feld}={mapped!r}")

            row[feld] = "" if mapped is None else mapped

        row["gehalt_min_eur"] = normalize_salary(rec.get("gehalt_min_eur"))
        row["skills_top3"] = normalize_skills(rec.get("skills_top3"))
        csv_rows.append(row)

    return csv_rows, mapped_count, problems

all_frontier_dfs = {}

for model, json_path in PREDICTION_FILES.items():
    if not json_path.exists():
        raise FileNotFoundError(
            f"Prediction-Datei für {model!r} fehlt: {json_path}\n"
            f"Lege sie als frontier_predictions_{model}.json in den Notebook-Ordner, Repo-Root oder annotation/."
        )

    records = extract_json_array(json_path)
    print(f"\n{model}: Records geladen: {len(records)} (erwartet: 12)")

    csv_rows, mapped_count, problems = records_to_csv_rows(records)
    out_path = FRONTIER_CSV_PATHS[model]

    with out_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=FELDER_CSV)
        writer.writeheader()
        writer.writerows(csv_rows)

    all_frontier_dfs[model] = pd.DataFrame(csv_rows)
    print(f"{model}: CSV geschrieben → {out_path}")
    print(f"{model}: Schema-Verletzungs-Mappings angewendet: {mapped_count}")
    if problems:
        print(f"{model}: WARNUNGEN")
        for p in problems:
            print(f"  - {p}")

print("\nTerminal-Checks, falls validate.py vorhanden ist:")
for model, csv_path in FRONTIER_CSV_PATHS.items():
    print(f"  python annotation/validate.py {csv_path}")
if GOLD_PATH.exists():
    for model, csv_path in FRONTIER_CSV_PATHS.items():
        print(f"  python annotation/validate.py {GOLD_PATH} --kappa-against {csv_path}")


In [ ]:
# κ-Compute: eigene Gold-Annotation ↔ ChatGPT / Claude
# Gleicher Grundalgorithmus wie validate.py, hier für beide Frontier-Dateien in einer Tabelle.

if gold_df is None:
    raise FileNotFoundError("Für den κ-Vergleich fehlt annotations/mein_gold.csv oder annotation/meine_gold.csv.")

def cohen_kappa(labels_a, labels_b):
    assert len(labels_a) == len(labels_b)
    n = len(labels_a)
    if n == 0:
        return float("nan")
    agreed = sum(1 for a, b in zip(labels_a, labels_b) if a == b)
    p_o = agreed / n
    cat_a, cat_b = Counter(labels_a), Counter(labels_b)
    p_e = sum((cat_a[c] / n) * (cat_b[c] / n) for c in set(cat_a) | set(cat_b))
    if p_e == 1.0:
        return float("nan")
    return (p_o - p_e) / (1 - p_e)

def norm_label(v):
    if pd.isna(v):
        return ""
    return str(v).strip()

mein_df = gold_df.copy()
if "refnr" not in mein_df.columns and "id" in mein_df.columns:
    mein_df["refnr"] = mein_df["id"]
mein_df["refnr"] = mein_df["refnr"].astype(str)
mein_by_id = {str(r["refnr"]): dict(r) for _, r in mein_df.iterrows()}

CAT_FELDER = ["homeoffice", "vertragsart", "erfahrungslevel"]

kappa_rows = []
disagreements = []

for model, csv_path in FRONTIER_CSV_PATHS.items():
    frontier_df = pd.read_csv(csv_path).rename(columns={"id": "refnr"})
    frontier_df["refnr"] = frontier_df["refnr"].astype(str)
    frontier_by_id = {str(r["refnr"]): dict(r) for _, r in frontier_df.iterrows()}

    # Reihenfolge: Gold-Reihenfolge, nicht alphabetisch
    common_ids = [rid for rid in gold_ids if rid in frontier_by_id]
    if not common_ids:
        print(f"Warnung: Keine gemeinsamen IDs für {model}.")
        continue

    for feld in CAT_FELDER:
        a = [norm_label(mein_by_id[r].get(feld)) for r in common_ids]
        b = [norm_label(frontier_by_id[r].get(feld)) for r in common_ids]
        k = cohen_kappa(a, b)
        agree = sum(1 for x, y in zip(a, b) if x == y)
        kappa_rows.append({
            "modell": model,
            "feld": feld,
            "κ": round(k, 3),
            "Übereinst.": f"{agree/len(common_ids):.0%}",
            "agree": agree,
            "n": len(common_ids),
        })
        for rid, x, y in zip(common_ids, a, b):
            if x != y:
                disagreements.append({
                    "modell": model,
                    "refnr": rid,
                    "feld": feld,
                    "ich": x,
                    "frontier": y,
                })

kappa_df = pd.DataFrame(kappa_rows)
disagreement_df = pd.DataFrame(disagreements)

print("κ Mensch ↔ Frontier:\n")
display(kappa_df)

print("\nInterpretation (Landis & Koch): < 0.40 mäßig · 0.41–0.60 moderat · 0.61–0.80 substanziell · > 0.80 fast perfekt")
print(f"\nDisagreements gesamt über beide Modelle und kategoriale Felder: {len(disagreement_df)}")

if not disagreement_df.empty:
    display(disagreement_df.sort_values(["modell", "feld", "refnr"]).reset_index(drop=True))



In [ ]:
# Direkte Gegenüberstellung: mein_gold ↔ ChatGPT ↔ Claude
# Zeigt alle Felder, in denen sich mindestens zwei der drei Quellen unterscheiden.

if gold_df is None:
    raise FileNotFoundError("Für die Gegenüberstellung fehlt annotations/mein_gold.csv oder annotation/meine_gold.csv.")
if not all(path.exists() for path in FRONTIER_CSV_PATHS.values()):
    raise FileNotFoundError("Bitte zuerst die JSON→CSV-Zelle ausführen.")

def norm_label_local(v):
    if pd.isna(v):
        return ""
    return str(v).strip()

mein_df = gold_df.copy()
if "refnr" not in mein_df.columns and "id" in mein_df.columns:
    mein_df["refnr"] = mein_df["id"]
mein_df["refnr"] = mein_df["refnr"].astype(str)

chatgpt_df = pd.read_csv(FRONTIER_CSV_PATHS["chatgpt"]).rename(columns={"id": "refnr"})
claude_df = pd.read_csv(FRONTIER_CSV_PATHS["claude"]).rename(columns={"id": "refnr"})
chatgpt_df["refnr"] = chatgpt_df["refnr"].astype(str)
claude_df["refnr"] = claude_df["refnr"].astype(str)

mein_by_id = {str(r["refnr"]): dict(r) for _, r in mein_df.iterrows()}
chatgpt_by_id = {str(r["refnr"]): dict(r) for _, r in chatgpt_df.iterrows()}
claude_by_id = {str(r["refnr"]): dict(r) for _, r in claude_df.iterrows()}

compare_fields = ["homeoffice", "vertragsart", "erfahrungslevel", "gehalt_min_eur", "gehalt_zeitraum", "skills_top3"]
common_ids_all = [rid for rid in gold_ids if rid in mein_by_id and rid in chatgpt_by_id and rid in claude_by_id]

comparison_rows = []
for rid in common_ids_all:
    for feld in compare_fields:
        human = norm_label_local(mein_by_id[rid].get(feld))
        chatgpt = norm_label_local(chatgpt_by_id[rid].get(feld))
        claude = norm_label_local(claude_by_id[rid].get(feld))
        if len({human, chatgpt, claude}) > 1:
            comparison_rows.append({
                "refnr": rid,
                "feld": feld,
                "mein_gold": human,
                "chatgpt": chatgpt,
                "claude": claude,
                "chatgpt_match": human == chatgpt,
                "claude_match": human == claude,
            })

comparison_df = pd.DataFrame(comparison_rows)

# Rückwärtskompatibel: reine ChatGPT-Claude-Differenzen wie vorher.
model_diff_rows = []
for rid in common_ids_all:
    for feld in compare_fields:
        a = norm_label_local(chatgpt_by_id[rid].get(feld))
        b = norm_label_local(claude_by_id[rid].get(feld))
        if a != b:
            model_diff_rows.append({
                "refnr": rid,
                "feld": feld,
                "chatgpt": a,
                "claude": b,
            })
model_diff_df = pd.DataFrame(model_diff_rows)

print(f"Gegenüberstellung mein_gold ↔ ChatGPT ↔ Claude: {len(comparison_df)} abweichende Feldzeilen")
if not comparison_df.empty:
    display(comparison_df.sort_values(["refnr", "feld"]).reset_index(drop=True))

print(f"\nDirekte ChatGPT-Claude-Differenzen: {len(model_diff_df)}")
if not model_diff_df.empty:
    display(model_diff_df.sort_values(["refnr", "feld"]).reset_index(drop=True))



## Ergebnisse des Frontier-Vergleichs

### JSON→CSV-Checks

**ChatGPT**

- Records geladen: 12 (erwartet: 12)
- CSV geschrieben → `annotation/frontier_gold_chatgpt.csv`
- Schema-Verletzungs-Mappings angewendet: 0

**Claude**

- Records geladen: 12 (erwartet: 12)
- CSV geschrieben → `annotation/frontier_gold_claude.csv`
- Schema-Verletzungs-Mappings angewendet: 0

### κ Mensch ↔ Frontier

| modell | feld | κ | Übereinst. | agree | n |
|---|---|---:|---:|---:|---:|
| chatgpt | homeoffice | 0.507 | 75% | 9 | 12 |
| chatgpt | vertragsart | 1.000 | 100% | 12 | 12 |
| chatgpt | erfahrungslevel | 0.526 | 75% | 9 | 12 |
| claude | homeoffice | 0.385 | 67% | 8 | 12 |
| claude | vertragsart | 1.000 | 100% | 12 | 12 |
| claude | erfahrungslevel | 0.363 | 50% | 6 | 12 |

Interpretation (Landis & Koch): < 0.40 mäßig · 0.41–0.60 moderat · 0.61–0.80 substanziell · > 0.80 fast perfekt

### Disagreements gegenüber deiner Handannotation

Disagreements gesamt über beide Modelle und kategoriale Felder: **16**

| modell | refnr | feld | ich | frontier |
|---|---|---|---|---|
| chatgpt | 11949-17215590-S | erfahrungslevel | nicht_genannt | junior |
| chatgpt | 13151-1570027-1-S | erfahrungslevel | mid | junior |
| chatgpt | 14225-2aaa34ba5c393d2a-S | erfahrungslevel | nicht_genannt | mid |
| chatgpt | 14225-2aaa34ba5c393d2a-S | homeoffice | teilweise | nein |
| chatgpt | 16724-0062809539-S | homeoffice | ja | nicht_genannt |
| chatgpt | 18896-8565435-S | homeoffice | teilweise | nicht_genannt |
| claude | 11949-17196786-S | erfahrungslevel | mid | senior |
| claude | 12265-489382_JB5131539-S | erfahrungslevel | mid | nicht_genannt |
| claude | 13151-1568687-1-S | erfahrungslevel | mid | junior |
| claude | 13151-1570027-1-S | erfahrungslevel | mid | junior |
| claude | 13644-307612-S | erfahrungslevel | mid | nicht_genannt |
| claude | 14036-0005680f46a001-S | erfahrungslevel | mid | nicht_genannt |
| claude | 13151-1570027-1-S | homeoffice | teilweise | nicht_genannt |
| claude | 14225-2aaa34ba5c393d2a-S | homeoffice | teilweise | nicht_genannt |
| claude | 16724-0062809539-S | homeoffice | ja | nicht_genannt |
| claude | 18896-8565435-S | homeoffice | teilweise | nicht_genannt |

### Direkte ChatGPT-Claude-Differenzen

Direkte ChatGPT-Claude-Differenzen: **14**

| refnr | feld | chatgpt | claude |
|---|---|---|---|
| 11949-17196786-S | erfahrungslevel | mid | senior |
| 11949-17196786-S | skills_top3 | Machine Learning\|Python\|statistische Modellierung | Python\|Machine Learning |
| 11949-17215590-S | erfahrungslevel | junior | nicht_genannt |
| 12265-489382_JB5131539-S | erfahrungslevel | mid | nicht_genannt |
| 12265-489382_JB5131539-S | skills_top3 | SQL\|ETL\|Power BI |  |
| 13151-1568687-1-S | erfahrungslevel | mid | junior |
| 13151-1570027-1-S | homeoffice | teilweise | nicht_genannt |
| 13644-307612-S | erfahrungslevel | mid | nicht_genannt |
| 14036-0005680f46a001-S | erfahrungslevel | mid | nicht_genannt |
| 14225-2aaa34ba5c393d2a-S | erfahrungslevel | mid | nicht_genannt |
| 14225-2aaa34ba5c393d2a-S | homeoffice | nein | nicht_genannt |
| 14225-2aaa34ba5c393d2a-S | skills_top3 | Markov Decision Processes\|Q-Learning\|Value Functions |  |
| 16724-0062809539-S | skills_top3 | Jedox\|Power BI\|Data Warehouse | Power BI\|Jedox\|Data Warehouse |
| 18896-8565435-S | skills_top3 | PowerBI\|SQL\|OLAP-Datenbanken | Power BI\|SQL\|OLAP |

### Terminal-Checks, falls `validate.py` vorhanden ist

```bash
python annotation/validate.py "annotation/frontier_gold_chatgpt.csv"
python annotation/validate.py "annotation/frontier_gold_claude.csv"
python annotation/validate.py "annotation/meine_gold.csv" --kappa-against "annotation/frontier_gold_chatgpt.csv"
python annotation/validate.py "annotation/meine_gold.csv" --kappa-against "annotation/frontier_gold_claude.csv"
```

### Drei konkrete Disagreement-Beispiele für das Memo

1. `16724-0062809539-S`, Feld `homeoffice`: Ich = `ja`, ChatGPT/Claude = `nicht_genannt`. Zu prüfen ist, ob in der Anzeige Homeoffice ohne Modalität erwähnt wurde oder ob die Modelle korrekt nichts gefunden haben.
2. `13151-1570027-1-S`, Feld `erfahrungslevel`: Ich = `mid`, ChatGPT/Claude = `junior`. Hier ist die Schema-Regel für Praktikum/Werkstudent vs. fachliche Aufgaben zu diskutieren.
3. `14225-2aaa34ba5c393d2a-S`, Feld `homeoffice`: Ich = `teilweise`, ChatGPT = `nein`, Claude = `nicht_genannt`. Hier ist entscheidend, ob „on-site“ als Präsenzpflicht/kein Homeoffice oder nur als Durchführungsort zu interpretieren ist.


In [ ]:
# Optional: Tabellen als CSV exportieren

EXPORT_DIR = ANNOTATION_DIR
if "kappa_df" in globals() and not kappa_df.empty:
    kappa_df.to_csv(EXPORT_DIR / "frontier_kappa_chatgpt_claude.csv", index=False, encoding="utf-8")
    print("geschrieben:", EXPORT_DIR / "frontier_kappa_chatgpt_claude.csv")

if "disagreement_df" in globals() and not disagreement_df.empty:
    disagreement_df.to_csv(EXPORT_DIR / "frontier_disagreements_vs_gold.csv", index=False, encoding="utf-8")
    print("geschrieben:", EXPORT_DIR / "frontier_disagreements_vs_gold.csv")

if "model_diff_df" in globals() and not model_diff_df.empty:
    model_diff_df.to_csv(EXPORT_DIR / "frontier_differences_chatgpt_vs_claude.csv", index=False, encoding="utf-8")
    print("geschrieben:", EXPORT_DIR / "frontier_differences_chatgpt_vs_claude.csv")

if "comparison_df" in globals() and not comparison_df.empty:
    comparison_df.to_csv(EXPORT_DIR / "frontier_comparison_mein_gold_chatgpt_claude.csv", index=False, encoding="utf-8")
    print("geschrieben:", EXPORT_DIR / "frontier_comparison_mein_gold_chatgpt_claude.csv")

